<a href="https://colab.research.google.com/github/davidlealo/sic_ia_2026/blob/main/clases/sic_2026_ia_Clase_24.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SIC 2026 · Inteligencia Artificial
## Clase 24: última revisión de avances antes de presentar

Jueves 26 de noviembre de 2026 · Sesión de revisión por equipos

Fue la última sesión para mostrar avances antes de las presentaciones. Los equipos mostraron sus prototipos y recibieron comentarios. Este notebook reúne el calendario, las ideas que salieron y tres pruebas que conviene hacer antes del lunes: **elegir el umbral**, **reentrenar con menos épocas** y **darle peso a la observación del usuario** en un RAG.

## 1. Calendario de cierre

| Fecha | Qué pasa |
|---|---|
| Sábado 28 de noviembre | **No hay clase.** Las dudas, por correo (es más rápido que Discord). |
| **Lunes 30 de noviembre y martes 1 de diciembre** | **Presentaciones del proyecto final.** La idea es que todos los equipos presenten esos días. |
| Jueves 3 de diciembre | Fecha máxima para presentar. |
| **10 de diciembre** | Plazo para el **informe final**. |

Si necesitan revisar algo puntual antes de presentar, escriban por correo y se coordina una llamada.

## 2. El informe: cuenten las decisiones, no solo los pasos

La estructura del informe es una **propuesta**: si una sección (por ejemplo, «flujo de trabajo» o «diseño del sistema») no calza con su proyecto, adáptenla.

- **Flujo de trabajo** es el camino de los datos: de dónde salen, dónde se guardan, cómo se procesan. Pero la lista de pasos («descargamos, limpiamos con pandas, entrenamos») es lo menos interesante de leer.
- Lo que más vale es **por qué tomaron cada decisión**: qué modelos probaron y por qué se quedaron con uno, qué problemas tuvieron (de datos, de hardware, de tiempo) y cómo los resolvieron, y cómo se repartieron el trabajo.
- **Diseño del sistema**: un diagrama vale más que párrafos. Si ya tienen un esquema de cómo se conectan las partes, pónganlo y agréguenle una explicación corta.

## 3. Ideas que salieron en la revisión

**Si la demo falla más que las métricas, algo está mal.** Un equipo con un clasificador de radiografías tenía métricas muy buenas (se equivocaba en menos del 2 % de los casos), pero en la demo se equivocó en 3 de 5 imágenes «normales». Estadísticamente es muy raro. Hay que buscar la causa antes de presentar:

1. **¿La demo carga el mismo modelo que se evaluó?** Revisen la ruta y la versión del archivo.
2. **¿La demo preprocesa las imágenes igual que el entrenamiento?** (tamaño, escala 0–1 o `preprocess_input`, canales). Es la causa más común.
3. **¿El umbral es el adecuado?** Con 0,5 había muchos falsos positivos de neumonía (sección 4).
4. **¿Está sobreajustado?** La pérdida de validación dejó de bajar después de la época 5 o 6 (sección 5).

**El mapa de calor ayuda a detectar el problema.** Con Grad-CAM se vio que, en algunas imágenes mal clasificadas, el modelo miraba zonas fuera de los pulmones. Es una pista de que aprendió algo que no corresponde.

**La observación del usuario se diluye.** Un equipo de reciclaje combina un clasificador de imágenes, un RAG con documentación y un modelo de lenguaje que redacta la recomendación. La observación del usuario («es un vaso plástico blanco») entraba al prompt junto con mucho otro contexto y casi no cambiaba la respuesta. Hay que darle peso explícito y **probarlo** (sección 6).

**Interfaz: lo importante, visible y completo.** Si un texto es importante, no puede quedar cortado ni depender de pasar el mouse por encima. Lo avanzado o secundario puede ir en desplegables («Opciones avanzadas»).

**Comparar contra datos que el modelo no vio.** Un equipo de reconocimiento de aves por audio armó su conjunto de prueba solo con grabaciones que no estaban en el entrenamiento de los modelos preentrenados, y con eso compara modelos generales contra uno especializado en especies de Chile. Solo filtrar las especies chilenas ya mejoró el acierto en unos 7 puntos.

**No todo va más rápido en GPU.** Un modelo pequeño puede correr mejor en CPU: mover los datos a la tarjeta gráfica toma más tiempo que hacer el cálculo. Además, pasos como calcular espectrogramas suelen correr en CPU.

## 4. Elegir el umbral con datos de validación

Un clasificador binario entrega una probabilidad. El umbral de 0,5 es solo un punto de partida: si hay muchos falsos positivos, subirlo puede ayudar, a costa de perder algunos verdaderos positivos.

Elijan el umbral con el conjunto de **validación** y después confirmen con el de prueba. La celda usa datos simulados: reemplacen `y_val` y `p_val` por sus etiquetas reales y las probabilidades de su modelo (`modelo.predict(X_val).ravel()`).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# Datos simulados: reemplacen por los suyos
rng = np.random.default_rng(0)
y_val = rng.integers(0, 2, 600)                       # 1 = neumonía, 0 = normal
p_val = np.clip(np.where(y_val == 1, rng.normal(0.75, 0.18, 600), rng.normal(0.40, 0.22, 600)), 0, 1)

filas = []
for umbral in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pred = (p_val >= umbral).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
    filas.append({"umbral": umbral, "falsos_positivos": fp, "falsos_negativos": fn,
                  "precision": round(precision_score(y_val, pred), 3),
                  "recall": round(recall_score(y_val, pred), 3),
                  "f1": round(f1_score(y_val, pred), 3)})
tabla = pd.DataFrame(filas)
print(tabla.to_string(index=False))
print("\nUmbral con mejor F1:", tabla.loc[tabla.f1.idxmax(), "umbral"])

En salud, un falso negativo (decir «normal» a alguien con neumonía) suele ser más grave que un falso positivo. Por eso no siempre conviene el umbral con mejor F1: pueden fijar un **recall mínimo** (por ejemplo, 0,95) y, entre los umbrales que lo cumplen, elegir el que tenga menos falsos positivos. Cuenten esa decisión en el informe.

Y usen **el mismo umbral** en las métricas y en la demo.

## 5. ¿Sobreajuste? Reentrenar con menos épocas y guardar otra versión

Si la pérdida de validación deja de bajar (o sube) mientras la de entrenamiento sigue bajando, el modelo está memorizando. Pruebas sugeridas en clase:

- Entrenar con **6 o 7 épocas** como máximo.
- `EarlyStopping` con `patience=3` (estaba en 5) y `restore_best_weights=True`.
- **No sobrescribir el modelo anterior**: guardar la nueva versión con otro nombre y comparar las dos con el mismo conjunto de prueba y el mismo umbral.

La celda es una plantilla para Keras: cambien `modelo_base`, `train_ds` y `val_ds` por los suyos.

In [ ]:
# Plantilla: descomenten y ajusten a su proyecto
# from tensorflow import keras
#
# modelo_v2 = keras.models.clone_model(modelo_base)       # misma arquitectura, pesos nuevos
# modelo_v2.compile(optimizer=keras.optimizers.Adam(1e-4), loss="binary_crossentropy",
#                   metrics=["accuracy", keras.metrics.AUC(name="auc")])
#
# parar = keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
# historia = modelo_v2.fit(train_ds, validation_data=val_ds, epochs=7, callbacks=[parar])
#
# modelo_v2.save("modelo_neumonia_v2.keras")              # el v1 queda intacto
#
# # Comparación justa: mismo conjunto de prueba, mismo umbral
# for nombre, m in [("v1", keras.models.load_model("modelo_neumonia_v1.keras")), ("v2", modelo_v2)]:
#     print(nombre, m.evaluate(test_ds, verbose=0))

## 6. Darle peso a la observación del usuario en un RAG

Si la observación del usuario va mezclada con la clasificación y varios documentos, el modelo de lenguaje tiende a ignorarla. Tres cosas ayudan:

1. **Separarla y nombrarla** en el prompt, no mezclarla con el resto.
2. **Decir qué hacer si contradice al clasificador**: por ejemplo, que la observación manda si describe algo que la foto no muestra (un forro de aluminio por dentro), y que la respuesta lo diga.
3. **Usarla también para buscar**: agregarla a la consulta del RAG, así los documentos recuperados ya consideran lo que dijo la persona.

Y sobre todo, **probarlo**: la misma imagen con y sin observación debería dar respuestas distintas cuando corresponde.

In [ ]:
def armar_prompt(clase, confianza, documentos, observacion=""):
    partes = [
        "Eres un asistente de reciclaje en Chile. Responde en español, en 3 a 5 líneas.",
        f"CLASIFICACIÓN DE LA IMAGEN: {clase} (confianza {confianza:.0%}).",
    ]
    if observacion:
        partes.append(
            "OBSERVACIÓN DE LA PERSONA (prioritaria): " + observacion + "\n"
            "Si la observación contradice la clasificación, confía en la observación cuando describa "
            "algo que la foto no puede mostrar (materiales internos, capas, uso previo) y dilo en la respuesta."
        )
    partes.append("DOCUMENTOS:\n" + "\n".join(f"- {d}" for d in documentos))
    partes.append("Recomienda qué hacer con el objeto. Si la información no alcanza, dilo.")
    return "\n\n".join(partes)

def consulta_rag(clase, observacion=""):
    # La observación también entra a la búsqueda de documentos
    return f"{clase}. {observacion}".strip()

docs = ["Vasos de plástico PS (número 6): en muchas comunas no se reciclan; revisar el punto limpio.",
        "Textiles: donar si están en buen estado o llevar a contenedores de ropa."]
print(consulta_rag("textil", "es un vaso plástico blanco con rayitas"))
print()
print(armar_prompt("textil", 0.98, docs, "es un vaso plástico blanco con rayitas"))

Prueben su sistema con dos o tres casos difíciles (los que hoy fallan) **con y sin observación**, y guarden las respuestas para mostrarlas en la presentación: es una buena forma de demostrar que el sistema escucha al usuario.

Si además quieren que el **clasificador** aprenda de las correcciones, guarden la imagen con la etiqueta corregida para un próximo reentrenamiento. Eso es otro paso y no es necesario para el lunes.

## Dudas

Por correo, que es más rápido que Discord. Nos vemos el lunes en las presentaciones.